# Cahn-Hilliard and Canham-Helfrich on a sphere


Energy functional:
$$
E(\varphi, h) = \int_\Omega \frac{1}{\varepsilon} W(\varphi) + \frac{\varepsilon}{2} |\nabla \varphi|^2 + \frac{\sigma}{2}|\nabla h|^2 + \frac{\kappa}{2}|\Delta h|^2 + \Lambda \varphi \Delta h \, d\mathcal{L}^d
$$

PDE:



In [ ]:
%%capture
try:
    import dolfin
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/fenics-install-release-real.sh" -O "/tmp/fenics-install.sh" && bash "/tmp/fenics-install.sh"
    import dolfin

In [ ]:
!git clone https://github.com/dpeschka/cahn-hilliard-sphere.git
folder = 'surface-sphere/'

In [ ]:
from dolfin import *
import logging
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

temp = Mesh('cahn-hilliard-sphere/data/sphere.xml')

mesh = BoundaryMesh(temp, 'exterior')
mesh = refine(mesh)

meshc = Mesh(mesh)
mesh = refine(mesh)

x = mesh.coordinates()
x[:] /= np.sqrt(np.sum(x**2, axis=1))[:, np.newaxis]

xc = meshc.coordinates()
xc[:] /= np.sqrt(np.sum(xc**2, axis=1))[:, np.newaxis]

V = VectorFunctionSpace(mesh, 'P', 1)
disp = Expression(("x[0]*0.5","0","0"),degree=2)
u = interpolate(disp,V)
ALE.move(mesh,u)

Vc = VectorFunctionSpace(meshc, 'P', 1)
uc = interpolate(disp,Vc)
ALE.move(meshc,uc)


logging.getLogger("FFC").setLevel(logging.ERROR)
logging.getLogger("UFL_LEGACY").setLevel(logging.ERROR)
logging.getLogger("UFL").setLevel(logging.ERROR)
set_log_level(LogLevel.ERROR)

parameters['linear_algebra_backend'] = 'PETSc'
parameters['reorder_dofs_serial'] = True
parameters['form_compiler']['quadrature_degree'] = 8
parameters['form_compiler']['cpp_optimize'] = True
parameters['form_compiler']['optimize'] = True

In [ ]:
# FE definitions
L = -0.3
Λ = 1.0
ε = 0.1
T = 1.0
G = 2.0
κ = 0.3


# α β γ δ ε ζ η θ ι κ λ μ ν ξ ο π ρ σ τ υ φ χ ψ ω

# mesh = IntervalMesh(128,0,L)
FE   = FiniteElement("P", mesh.ufl_cell(), 1)   # scalar element
Q    = FunctionSpace(mesh,MixedElement([FE,FE,FE,FE,FE]))# mixed space


def W(φ):
  return 18*Λ*(φ*(1-φ))**2

# energy
def energy(q):
    φ,h,μ,μh,g = split(q)
    E  = ε/2 * inner(grad(φ),grad(φ))*dx
    E += G/2 * inner(grad(h),grad(h))*dx
    E += κ/2 * g**2*dx
    E -= L * inner(grad(h),grad(φ))*dx
    E += 1/ε * W(φ)*dx
    return E


# single time step
def evolve(old_q, τ):
    # set up function spaces
    q,v = Function(Q),TestFunction(Q)
    φ,h,μ,μh,g = split(q)
    vφ,vh,vμ,vμh,vg = split(v)
    old_φ,old_h,_,_,_ = split(old_q)

    # define energy
    E = energy(q)
    m  = 1.0
    mh = 1.0

    # define weak form
    Res  = (μ*vφ+μh*vh)*dx - derivative(E, q, v)
    Res += (g*vg - inner(grad(h),grad(vg)))*dx
    Res += ε**2*m*τ*inner(grad(μ),grad(vμ))*dx + vμ*(φ-old_φ)*dx
    Res += ε**2*mh*τ*inner(grad(μh),grad(vμh))*dx + vμh*(h-old_h)*dx
    bc = []

    Jac     = derivative(Res, q)

    problem = NonlinearVariationalProblem(Res, q, bc, Jac)
    solver  = NonlinearVariationalSolver(problem)

    prm = solver.parameters
    prm['newton_solver']['error_on_nonconvergence'] = False
    prm['newton_solver']['report'] = False
    prm['newton_solver']['absolute_tolerance'] = 1e-5
    prm['newton_solver']['relative_tolerance'] = 1e-5

    q.assign(old_q)
    iterations, converged = solver.solve()

    return q,iterations,converged

# initial data
idata = Expression(("((float)(rand()))","0","0","0","0"),degree=2,ε=ε)
old_q = interpolate(idata,Q)
old_q,it,conv = evolve(old_q,1e-6)
print('init:',it,conv)
times = []
energies = []
sols = []

# initial time stepping, later adaptive
t = 0
τ = 0.3e-4
n_steps = 1000
dt = Constant(τ)

f_phi = File('data/sol_phi.pvd')
f_h   = File('data/sol_h.pvd')

for i in tqdm(range(n_steps)):
  dt.assign(τ)
  q,it,conv = evolve(old_q, dt)

  if conv:
    phi,h,mu,muh,g = q.split()
    phi.rename("phi","phi")
    h.rename("h","h")

    f_phi << (phi,t)
    f_h << (h,t)

    t += τ
    old_q.assign(q)
    if (i % 2 == 0):
      times.append(t)
      sols.append(q)
    if it < 3:
      τ *= 1.1
    if it > 4:
      τ *= 0.95
  else:
    τ *= 0.75
  if t>T:
    break

In [ ]:
!zip data.zip data/*

In [ ]:
U = VectorFunctionSpace(mesh,'CG',2)
U1 = VectorFunctionSpace(mesh,'CG',1)

u = Function(U)
v = TestFunction(U)

#u1 = Function(U1)
#v1 = TestFunction(U1)


x = MeshCoordinates(mesh)

Res = inner(u,v)*dx - inner(grad(x),grad(v))*dx

#Res1 = inner(u1,v1)*dx - inner(u,v1)*dx

solve(Res==0,u)
# solve(Res1==0,u1)

u1 = project(u,U1)

f = File('curvature.pvd')
u1.rename('curv','curv')
f << u1

In [ ]:
temp = Mesh('cahn-hilliard-sphere/data/sphere.xml')

mesh = BoundaryMesh(temp, 'exterior')
x = mesh.coordinates()
x[:] /= np.sqrt(np.sum(x**2, axis=1))[:, np.newaxis]

u = project(Constant(1.0),FunctionSpace(mesh,'CG',1))
print(assemble(u*dx))
print(4*np.pi)

In [ ]:
from dolfin import *
import numpy as np

# 1. Linear sphere mesh
temp = Mesh("cahn-hilliard-sphere/data/sphere.xml")
mesh = BoundaryMesh(temp, "exterior")
#mesh = refine(mesh)
# project vertices onto exact sphere
x = mesh.coordinates()
x[:] /= np.sqrt(np.sum(x**2, axis=1))[:, np.newaxis]

# 2. P2 deformation field Phi
V2 = VectorFunctionSpace(mesh, "CG", 2)
Phi = interpolate(Expression(("x[0]", "x[1]", "x[2]"), degree=2), V2)

# 3. Project ALL P2 support points onto sphere
Phi_vec = Phi.vector().get_local()
coords = V2.tabulate_dof_coordinates()

for i in range(len(coords)):
    Y = coords[i] / np.linalg.norm(coords[i])
    Phi_vec[i] = Y[i % 3]

Phi.vector().set_local(Phi_vec)
Phi.vector().apply("insert")



F = grad(Phi)

t1 = F[:, 0]
t2 = F[:, 1]

J_surf = sqrt(inner(cross(t1, t2), cross(t1, t2)))

area = assemble(J_surf * dx(domain=mesh))

print("Improved area =", 2*area)
print("Exact area    =", 4*np.pi)
print("Error         =", abs(2*area - 4*np.pi))